# Análisis Exploratorio de Datos — Airbnb Madrid vs Milán

## Introducción

Este notebook realiza un **análisis exploratorio de datos (EDA)** sobre los listados de Airbnb en **Madrid** y **Milán**, dos de las principales ciudades turísticas de Europa.

### Objetivo

El objetivo es **conocer y comprender los datos en profundidad** antes de realizar cualquier transformación o modelado. Buscamos responder preguntas como:

- ¿Cómo es la estructura de los datos? ¿Qué columnas tenemos y de qué tipo?
- ¿Existen valores nulos, duplicados o errores evidentes?
- ¿Cómo se distribuyen los precios en cada ciudad?
- ¿Qué tipos de alojamiento predominan?
- ¿Qué barrios concentran los precios más altos?
- ¿Qué relación existe entre las variables numéricas?

### Enfoque

El análisis es **puramente exploratorio y no destructivo**: los datos se leen tal como están, sin aplicar ninguna transformación permanente. Los valores nulos se detectan pero no se tratan, y los outliers se filtran únicamente a efectos visuales. Todas las decisiones de limpieza se tomarán en el notebook `data_cleaning.ipynb`, con la información obtenida aquí como base.

### Fuente de datos

Los datos provienen de [Inside Airbnb](http://insideairbnb.com/), una fuente pública que publica periódicamente scrapes de los listados activos de Airbnb en ciudades de todo el mundo. Los archivos utilizados son:
- `madrid_airbnb.csv`
- `milan_airbnb.csv`

## 1. Importar bibliotecas

Cargamos las librerías necesarias para el análisis:

- **pandas**: manipulación y análisis de datos tabulares (lectura de CSV, agrupaciones, filtros)
- **numpy**: operaciones numéricas y cálculo de estadísticos
- **matplotlib**: visualización de bajo nivel, permite un control preciso de los gráficos
- **seaborn**: visualización estadística de alto nivel, construida sobre matplotlib

También establecemos una configuración visual global para que todos los gráficos del notebook tengan un estilo consistente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración general de gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '{:.2f}'.format)

print(' Librerías cargadas correctamente')

## 2. Cargar los datos

Leemos los dos archivos CSV desde la carpeta `data/raw/`. Además añadimos una columna `ciudad` a cada dataset para poder distinguirlos cuando los unamos en un único DataFrame combinado.

In [ ]:
madrid = pd.read_csv('../data/raw/madrid_airbnb.csv')
milan  = pd.read_csv('../data/raw/milan_airbnb.csv')

# Añadimos una columna para identificar la ciudad
madrid['ciudad'] = 'Madrid'
milan['ciudad']  = 'Milán'

# Dataset combinado
df = pd.concat([madrid, milan], ignore_index=True)

print(f'Madrid: {madrid.shape}  |  Milán: {milan.shape}  |  Total: {df.shape}')

## 3. Explorar las primeras filas

Un primer vistazo a las filas iniciales de cada dataset nos permite ver el formato de los datos, los nombres de las columnas y los valores típicos. Esto es útil para detectar rápidamente problemas evidentes como columnas mal nombradas o valores inesperados.

In [ ]:
print('=== MADRID ===')
display(madrid.head())
print('\n=== MILÁN ===')
display(milan.head())

## 4. Información general del dataset

`info()` nos proporciona un resumen técnico esencial del dataset:

- **Número de filas y columnas**: nos da el tamaño del problema
- **Tipo de dato de cada columna** (`int64`, `float64`, `object`...): nos indica si hay conversiones necesarias
- **Valores no nulos por columna**: de forma indirecta, nos muestra cuántos nulos hay

Este paso es especialmente relevante para detectar columnas con **tipos incorrectos**. Por ejemplo, es habitual que `price` venga como `object` (texto) si contiene símbolos como `$` o comas, lo que impediría hacer cálculos numéricos directamente.

In [ ]:
print('=== MADRID ===')
madrid.info()
print('\n=== MILÁN ===')
milan.info()

## 5. Estadísticas descriptivas

`describe()` genera un resumen estadístico automático de todas las variables numéricas:

| Estadístico | Qué nos dice |
|---|---|
| `count` | Número de valores no nulos (detecta nulos indirectamente) |
| `mean` | Media aritmética |
| `std` | Desviación típica (dispersión de los datos) |
| `min` / `max` | Valores extremos (útil para detectar outliers) |
| `25%` / `50%` / `75%` | Percentiles (distribución general) |

Comparar estos valores entre Madrid y Milán nos permite detectar diferencias estructurales entre ambas ciudades antes de visualizar nada.

In [ ]:
print('=== MADRID ===')
display(madrid.describe())
print('\n=== MILÁN ===')
display(milan.describe())

## 6. Valores nulos

Los valores nulos son uno de los problemas más comunes en datasets reales. Su presencia puede:

- **Sesgar** estadísticos como la media o la correlación
- **Provocar errores** en modelos de machine learning si no se tratan
- Indicar **problemas en la recogida de datos** (campos opcionales, errores de scraping, etc.)

Visualizamos la cantidad de nulos por columna en cada ciudad para identificar de un vistazo qué columnas requieren atención. En Airbnb, las columnas que más frecuentemente contienen nulos son `reviews_per_month` y `last_review` (ausentes cuando un alojamiento no tiene reseñas aún).

> **Importante:** En este notebook únicamente **detectamos y cuantificamos** los nulos. Las decisiones sobre cómo tratarlos (eliminar filas, imputar con la media, rellenar con 0, etc.) se tomarán en el notebook de limpieza `data_cleaning.ipynb`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, dataset, nombre in zip(axes, [madrid, milan], ['Madrid', 'Milán']):
    nulos = dataset.isnull().sum()
    nulos = nulos[nulos > 0]
    if nulos.empty:
        ax.text(0.5, 0.5, 'Sin valores nulos', ha='center', va='center', fontsize=13)
    else:
        nulos.plot(kind='bar', ax=ax, color='salmon', edgecolor='black')
    ax.set_title(f'Valores nulos — {nombre}')
    ax.set_ylabel('Número de nulos')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Distribución de variables numéricas

Analizamos visualmente cómo se distribuyen las principales variables numéricas. Las visualizaciones nos permiten detectar asimetrías, outliers y diferencias entre ciudades de un solo vistazo.

### 7.1 Distribución del precio (filtrando outliers)

Comparamos la distribución del precio por noche en Madrid y Milán. Filtramos el percentil 99 para eliminar valores extremos que distorsionarían la escala del gráfico. Esto es solo a efectos **visuales**, el dataset original no se modifica.

In [ ]:
# Filtramos outliers extremos (precio < percentil 99)
p99 = df['price'].quantile(0.99)
df_filt = df[df['price'] <= p99]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
sns.histplot(data=df_filt, x='price', hue='ciudad', kde=True,
             bins=50, ax=axes[0], palette=['#e63946', '#457b9d'])
axes[0].set_title('Distribución del precio por ciudad')
axes[0].set_xlabel('Precio (€/noche)')

# Boxplot
sns.boxplot(data=df_filt, x='ciudad', y='price', ax=axes[1],
            palette=['#e63946', '#457b9d'])
axes[1].set_title('Boxplot del precio por ciudad')
axes[1].set_ylabel('Precio (€/noche)')

plt.tight_layout()
plt.show()

print(df_filt.groupby('ciudad')['price'].describe().round(2))

### 7.2 Tipos de habitación

Airbnb clasifica los alojamientos en varias categorías: *Entire home/apt*, *Private room*, *Shared room* y *Hotel room*. Comparamos cómo se distribuyen en cada ciudad para entender la oferta disponible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dataset, nombre, color in zip(
    axes, [madrid, milan], ['Madrid', 'Milán'], ['#e63946', '#457b9d']
):
    counts = dataset['room_type'].value_counts()
    counts.plot(kind='bar', ax=ax, color=color, edgecolor='black')
    ax.set_title(f'Tipos de habitación — {nombre}')
    ax.set_ylabel('Número de anuncios')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 7.3 Top 10 barrios por precio medio

Identificamos los barrios más caros de cada ciudad según el precio medio por noche. Esto nos da una idea de las zonas premium y puede ser relevante para análisis geoespaciales posteriores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dataset, nombre, color in zip(
    axes, [madrid, milan], ['Madrid', 'Milán'], ['#e63946', '#457b9d']
):
    top = (
        dataset.groupby('neighbourhood')['price']
        .mean()
        .sort_values(ascending=False)
        .head(10)
    )
    top.plot(kind='barh', ax=ax, color=color, edgecolor='black')
    ax.invert_yaxis()
    ax.set_title(f'Top 10 barrios (precio medio) — {nombre}')
    ax.set_xlabel('Precio medio (€/noche)')

plt.tight_layout()
plt.show()

### 7.4 Matriz de correlación (variables numéricas)

La matriz de correlación de Pearson mide la **relación lineal** entre pares de variables numéricas. Los valores van de **-1 a 1**:

- **Cercano a 1**: correlación positiva fuerte — cuando una variable sube, la otra también tiende a subir
- **Cercano a -1**: correlación negativa fuerte — cuando una sube, la otra baja
- **Cercano a 0**: sin relación lineal aparente entre las dos variables

Este análisis nos permite:
- Detectar variables **redundantes** entre sí (multicolinealidad), lo que es importante en modelos de regresión
- Identificar qué variables tienen **mayor relación con el precio**, orientando la selección de features en fases posteriores de modelado

In [ ]:
num_cols = ['price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, dataset, nombre in zip(axes, [madrid, milan], ['Madrid', 'Milán']):
    corr = dataset[num_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                ax=ax, linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title(f'Correlación — {nombre}')

plt.tight_layout()
plt.show()

## 8. Conclusiones del análisis exploratorio

Este EDA nos ha permitido obtener una visión completa del estado de los datos y de las características principales de la oferta de Airbnb en Madrid y Milán. A continuación se resumen los hallazgos más relevantes:

### Calidad de los datos
- Se han detectado **valores nulos** en al menos las columnas `reviews_per_month` y `last_review`, que están ausentes en alojamientos sin reseñas. Será necesario decidir si imputar (con `0`) o eliminar estas filas.
- La columna `price` puede requerir **conversión de tipo** si viene como texto (`object`), lo que impediría operaciones numéricas directas.
- Existen **outliers significativos** en `price` y posiblemente en `minimum_nights`, que podrían distorsionar análisis y modelos si no se gestionan.

### Diferencias entre ciudades
- Los precios en ambas ciudades muestran una **distribución muy asimétrica**, con una cola larga hacia valores altos y una mediana probablemente inferior a la media.
- El tipo de alojamiento dominante es **Entire home/apt** en ambas ciudades, aunque la proporción entre tipos puede variar.
- Los **barrios más caros** coinciden en general con las zonas céntricas y turísticas de cada ciudad, lo que sugiere que la ubicación es un factor determinante en el precio.
- Las correlaciones entre variables numéricas son en general **débiles**, lo que indica que el precio no depende linealmente de una sola variable, sino de una combinación de factores.

### Próximos pasos

| Paso | Notebook |
|---|---|
| Limpieza: nulos, tipos, duplicados, outliers | `data_cleaning.ipynb` |
| Análisis avanzado y visualizaciones geoespaciales | `eda2.ipynb` |
